# Pipeline de Dados ANAC - Malha Aérea Brasileira

Este notebook realiza:
1. Download de dados mensais da ANAC desde 2000
2. Processamento e consolidação dos dados
3. Criação de Delta table no formato raw
4. Monitoramento de progresso do pipeline


In [ ]:
# Importar bibliotecas necessárias
import requests
import pandas as pd
from datetime import datetime, timedelta
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
import time
from tqdm import tqdm
import io
import zipfile


## Parâmetros de Configuração


In [ ]:
# Configurações
CATALOG = "main"  # Ajuste conforme seu catálogo
SCHEMA = "raw"    # Ajuste conforme seu schema
TABLE_NAME = "anac_malha_aerea_brasil"

# URL base da ANAC (ajuste conforme o link específico fornecido)
# Nota: A URL exata depende da estrutura do site da ANAC
BASE_URL = "https://www.gov.br/anac/pt-br/assuntos/dados-e-estatisticas/dados-estatisticos/arquivos"

# Período de dados
START_YEAR = 2000
END_YEAR = datetime.now().year
END_MONTH = datetime.now().month

# Diretório temporário para downloads
TEMP_DIR = "/tmp/anac_data"


## Funções Auxiliares


In [ ]:
def create_temp_directory():
    """Cria diretório temporário para downloads"""
    dbutils.fs.mkdirs(f"file:{TEMP_DIR}")
    print(f"✓ Diretório temporário criado: {TEMP_DIR}")

def generate_date_range(start_year, end_year, end_month):
    """Gera lista de anos e meses para download"""
    dates = []
    for year in range(start_year, end_year + 1):
        max_month = end_month if year == end_year else 12
        for month in range(1, max_month + 1):
            dates.append((year, month))
    return dates

def download_anac_data(year, month):
    """
    Baixa dados da ANAC para um mês específico
    
    Nota: Esta função precisa ser ajustada conforme a estrutura 
    exata da URL do site da ANAC
    """
    try:
        # Formato típico de URL da ANAC (ajustar conforme necessário)
        # Exemplo: dados-{ano}-{mes}.csv ou similar
        url = f"{BASE_URL}/dados-{year}-{month:02d}.csv"
        
        response = requests.get(url, timeout=30)
        
        if response.status_code == 200:
            # Tentar ler como CSV
            try:
                df = pd.read_csv(io.StringIO(response.text), encoding='latin-1', sep=';')
                return df, None
            except Exception as e:
                return None, f"Erro ao processar CSV: {str(e)}"
        else:
            return None, f"Status code: {response.status_code}"
            
    except Exception as e:
        return None, f"Erro no download: {str(e)}"

def create_progress_tracker(total_items):
    """Cria um tracker de progresso"""
    return {
        'total': total_items,
        'completed': 0,
        'successful': 0,
        'failed': 0,
        'start_time': time.time()
}

def update_progress(tracker, success=True):
    """Atualiza o progresso"""
    tracker['completed'] += 1
    if success:
        tracker['successful'] += 1
    else:
        tracker['failed'] += 1
    
    # Calcular estatísticas
    elapsed = time.time() - tracker['start_time']
    progress_pct = (tracker['completed'] / tracker['total']) * 100
    
    # Estimar tempo restante
    if tracker['completed'] > 0:
        avg_time_per_item = elapsed / tracker['completed']
        remaining_items = tracker['total'] - tracker['completed']
        eta_seconds = avg_time_per_item * remaining_items
        eta_str = time.strftime('%H:%M:%S', time.gmtime(eta_seconds))
    else:
        eta_str = "Calculando..."
    
    return {
        'progress_pct': progress_pct,
        'elapsed': time.strftime('%H:%M:%S', time.gmtime(elapsed)),
        'eta': eta_str,
        'successful': tracker['successful'],
        'failed': tracker['failed']
}

def display_progress(stats):
    """Exibe progresso formatado"""
    print(f"\r[{'=' * int(stats['progress_pct'] / 2)}{' ' * (50 - int(stats['progress_pct'] / 2))}] "
          f"{stats['progress_pct']:.1f}% | "
          f"✓ {stats['successful']} | "
          f"✗ {stats['failed']} | "
          f"Tempo: {stats['elapsed']} | "
          f"ETA: {stats['eta']}", end='')


## Pipeline Principal


### Etapa 1: Preparação


In [ ]:
print("=" * 80)
print("PIPELINE ANAC - MALHA AÉREA BRASILEIRA")
print("=" * 80)
print()

# Criar diretório temporário
create_temp_directory()

# Gerar lista de datas
date_list = generate_date_range(START_YEAR, END_YEAR, END_MONTH)
print(f"✓ Total de períodos para processar: {len(date_list)}")
print(f"  Período: {START_YEAR}/01 até {END_YEAR}/{END_MONTH:02d}")
print()


### Etapa 2: Download dos Dados


In [ ]:
print("Iniciando download dos dados...")
print("-" * 80)

# Inicializar tracker de progresso
tracker = create_progress_tracker(len(date_list))

# Lista para armazenar DataFrames
all_dataframes = []
failed_downloads = []

# Loop de download com progresso
for year, month in date_list:
    df, error = download_anac_data(year, month)
    
    if df is not None:
        # Adicionar colunas de metadados
        df['ano_referencia'] = year
        df['mes_referencia'] = month
        df['data_carga'] = datetime.now()
        all_dataframes.append(df)
        
        # Atualizar progresso (sucesso)
        stats = update_progress(tracker, success=True)
    else:
        # Registrar falha
        failed_downloads.append({
            'year': year,
            'month': month,
            'error': error
        })
        
        # Atualizar progresso (falha)
        stats = update_progress(tracker, success=False)
    
    # Exibir progresso
    display_progress(stats)
    
    # Pequeno delay para não sobrecarregar o servidor
    time.sleep(0.1)

print("\n")
print("-" * 80)
print(f"✓ Download concluído!")
print(f"  Sucessos: {tracker['successful']}")
print(f"  Falhas: {tracker['failed']}")
print()


### Etapa 3: Consolidação dos Dados


In [ ]:
if len(all_dataframes) > 0:
    print("Consolidando dados...")
    
    # Concatenar todos os DataFrames
    df_consolidated = pd.concat(all_dataframes, ignore_index=True)
    
    print(f"✓ Dados consolidados")
    print(f"  Total de registros: {len(df_consolidated):,}")
    print(f"  Total de colunas: {len(df_consolidated.columns)}")
    print(f"  Período: {df_consolidated['ano_referencia'].min()}/{df_consolidated['mes_referencia'].min():02d} "
          f"até {df_consolidated['ano_referencia'].max()}/{df_consolidated['mes_referencia'].max():02d}")
    print()
    
    # Exibir amostra dos dados
    print("Amostra dos dados:")
    display(df_consolidated.head(10))
    
else:
    raise Exception("Nenhum dado foi baixado com sucesso!")


### Etapa 4: Conversão para Spark DataFrame


In [ ]:
print("Convertendo para Spark DataFrame...")

# Converter Pandas para Spark
spark_df = spark.createDataFrame(df_consolidated)

# Otimizar tipos de dados
spark_df = spark_df.withColumn("ano_referencia", col("ano_referencia").cast("int"))
spark_df = spark_df.withColumn("mes_referencia", col("mes_referencia").cast("int"))
spark_df = spark_df.withColumn("data_carga", col("data_carga").cast("timestamp"))

print(f"✓ Conversão concluída")
print(f"  Partições: {spark_df.rdd.getNumPartitions()}")
print()

# Exibir schema
print("Schema da tabela:")
spark_df.printSchema()


### Etapa 5: Criação da Delta Table


In [ ]:
print("Criando Delta Table...")
print("-" * 80)

# Nome completo da tabela
full_table_name = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"

# Criar schema se não existir
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"✓ Schema verificado: {CATALOG}.{SCHEMA}")

# Salvar como Delta Table
spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("ano_referencia", "mes_referencia") \
    .saveAsTable(full_table_name)

print(f"✓ Delta Table criada: {full_table_name}")
print(f"  Particionamento: ano_referencia, mes_referencia")
print()


### Etapa 6: Validação e Estatísticas


In [ ]:
print("Validando dados carregados...")
print("-" * 80)

# Ler a tabela criada
df_validation = spark.table(full_table_name)

# Estatísticas básicas
total_records = df_validation.count()
total_partitions = df_validation.select("ano_referencia", "mes_referencia").distinct().count()

print(f"✓ Validação concluída")
print(f"  Total de registros: {total_records:,}")
print(f"  Total de partições: {total_partitions}")
print()

# Estatísticas por ano
print("Registros por ano:")
df_validation.groupBy("ano_referencia") \
    .count() \
    .orderBy("ano_referencia") \
    .show(100, truncate=False)

# Informações da tabela
print("Informações da tabela Delta:")
spark.sql(f"DESCRIBE DETAIL {full_table_name}").show(truncate=False)


### Etapa 7: Relatório de Falhas (se houver)


In [ ]:
if len(failed_downloads) > 0:
    print("⚠ RELATÓRIO DE FALHAS")
    print("-" * 80)
    print(f"Total de períodos com falha: {len(failed_downloads)}")
    print()
    
    # Criar DataFrame com falhas
    df_failures = pd.DataFrame(failed_downloads)
    display(df_failures)
    
    print()
    print("Nota: Você pode tentar reprocessar estes períodos manualmente")
else:
    print("✓ Nenhuma falha registrada - todos os períodos foram processados com sucesso!")


## Resumo Final


In [ ]:
print("=" * 80)
print("PIPELINE CONCLUÍDO COM SUCESSO!")
print("=" * 80)
print()
print(f"📊 Tabela criada: {full_table_name}")
print(f"📈 Total de registros: {total_records:,}")
print(f"📅 Período: {START_YEAR} até {END_YEAR}")
print(f"✓ Sucessos: {tracker['successful']}/{tracker['total']}")
print(f"✗ Falhas: {tracker['failed']}/{tracker['total']}")
print(f"⏱ Tempo total: {stats['elapsed']}")
print()
print("Para consultar os dados:")
print(f"  SELECT * FROM {full_table_name} LIMIT 10;")
print()
print("=" * 80)


## Próximos Passos

1. **Análise Exploratória**: Execute queries para entender os dados
2. **Transformações**: Crie tabelas silver/gold com regras de negócio
3. **Visualizações**: Crie dashboards para análise da malha aérea
4. **Automação**: Configure um job para atualização mensal automátiac

### Exemplo de Query Analítica

```sql
SELECT 
    ano_referencia,
    mes_referencia,
    COUNT(*) as total_voos,
    COUNT(DISTINCT empresa) as total_empresas,
    COUNT(DISTINCT aeroporto_origem) as total_aeroportos
FROM {full_table_name}
GROUP BY ano_referencia, mes_referencia
ORDER BY ano_referencia DESC, mes_referencia DESC
LIMIT 12;
```
